# WORKSPACE-CONTEXT

> **Purpose**: Self-documentation for this workspace. Auto-discovers items and schemas, documents conventions and ownership, outputs structured context for AI assistants.
> 
> **How to use**: Run all cells to refresh auto-discovered metadata. Update markdown cells when conventions, ownership, or architecture change.

---

## Workspace Identity

| Property | Value |
|----------|-------|
| **Workspace** | `YOUR_WORKSPACE_NAME` |
| **Stage** | Development / Test / Production |
| **Capacity** | F64 - capacity-name |
| **Owner** | @your-name (Team Name) |
| **Team** | Your Team Name |
| **Git Repo** | `org/repo-name` (branch: `main`) |
| **Related Workspaces** | DEV: this -> TEST: workspace-test -> PROD: workspace-prod |
| **Last Reviewed** | YYYY-MM-DD |

### Purpose

Describe what this workspace is for in 1-2 sentences.
Example: *This workspace contains the sales analytics data platform: bronze -> silver -> gold lakehouse layers feeding Power BI reports.*


In [ ]:
# --- Auto-Discover Workspace Identity ---
import sempy.fabric as fabric

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "unknown")
workspace_name = notebookutils.runtime.context.get("currentWorkspaceName", "unknown")

print(f"Workspace: {workspace_name}")
print(f"Workspace ID: {workspace_id}")

from datetime import datetime, timezone
last_refreshed = datetime.now(timezone.utc).isoformat()
print(f"Last refreshed: {last_refreshed}")


In [ ]:
# --- Workspace Settings (auto-discovered via Fabric REST API) ---
import requests

token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
headers = {"Authorization": f"Bearer {token}"}
base_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# Workspace properties
ws_info = {}
try:
    r = requests.get(base_url, headers=headers)
    if r.status_code == 200:
        ws_info = r.json()
except Exception as e:
    print(f"Could not fetch workspace info: {e}")

# Spark settings
spark_info = {}
try:
    r = requests.get(f"{base_url}/spark/settings", headers=headers)
    if r.status_code == 200:
        spark_info = r.json()
except Exception as e:
    print(f"Could not fetch Spark settings: {e}")

# Git connection
git_info = {}
try:
    r = requests.get(f"{base_url}/git/connection", headers=headers)
    if r.status_code == 200:
        git_info = r.json()
except Exception as e:
    print(f"Could not fetch Git connection: {e}")

# Display summary
endpoints = ws_info.get("oneLakeEndpoints", {})
pool = spark_info.get("pool", {}).get("defaultPool", {})
env = spark_info.get("environment", {})
job = spark_info.get("job", {})
hc = spark_info.get("highConcurrency", {})
git_details = git_info.get("gitProviderDetails", {})
git_sync = git_info.get("gitSyncDetails", {})

print("=== Workspace Properties ===")
print(f"  Name:           {ws_info.get('displayName', 'N/A')}")
print(f"  Description:    {ws_info.get('description', 'N/A')}")
print(f"  Capacity ID:    {ws_info.get('capacityId', 'N/A')}")
print(f"  Region:         {ws_info.get('capacityRegion', 'N/A')}")
print(f"  OneLake Blob:   {endpoints.get('blobEndpoint', 'N/A')}")
print(f"  OneLake DFS:    {endpoints.get('dfsEndpoint', 'N/A')}")

print()
print("=== Spark Settings ===")
print(f"  Default Pool:   {pool.get('name', 'N/A')} ({pool.get('type', 'N/A')})")
print(f"  Environment:    {env.get('name', 'N/A')}")
print(f"  Runtime:        {env.get('runtimeVersion', 'N/A')}")
print(f"  Session Timeout: {job.get('sessionTimeoutInMinutes', 'N/A')} min")
print(f"  High Concurrency: {hc.get('notebookInteractiveRunEnabled', 'N/A')}")

print()
print("=== Git Connection ===")
print(f"  State:          {git_info.get('gitConnectionState', 'N/A')}")
print(f"  Provider:       {git_details.get('gitProviderType', 'N/A')}")
if git_details.get('gitProviderType') == 'AzureDevOps':
    print(f"  Organization:   {git_details.get('organizationName', 'N/A')}")
    print(f"  Project:        {git_details.get('projectName', 'N/A')}")
elif git_details.get('gitProviderType') == 'GitHub':
    print(f"  Owner:          {git_details.get('ownerName', 'N/A')}")
print(f"  Repository:     {git_details.get('repositoryName', 'N/A')}")
print(f"  Branch:         {git_details.get('branchName', 'N/A')}")
print(f"  Directory:      {git_details.get('directoryName', 'N/A')}")
if git_sync:
    print(f"  Head:           {git_sync.get('head', 'N/A')}")
    print(f"  Last Sync:      {git_sync.get('lastSyncTime', 'N/A')}")


In [ ]:
# --- Item Inventory (auto-discovered) ---
items = fabric.list_items()
print(f"Total items: {len(items)}")
print()

# Group by type for readability
for item_type in sorted(items["Type"].unique()):
    type_items = items[items["Type"] == item_type]
    print(f"\n=== {item_type} ({len(type_items)}) ===")
    for _, row in type_items.iterrows():
        desc = row.get("Description", "") or ""
        desc_str = f"  \u2014 {desc}" if desc else ""
        print(f"  {row['Display Name']}{desc_str}")


## Projects

A workspace often contains items belonging to multiple projects.
Document each project below so developers and AI assistants understand
what exists, how items relate, and what domain rules apply.

### Project 1: *(Project Name)*

| Field | Details |
|-------|--------|
| **Purpose** | What business problem does this project solve? |
| **Owner** | @name or team alias |
| **Status** | active / maintenance / deprecated |

**Items in this project:**

| Item | Type | Role |
|------|------|------|
| `bronze_lakehouse` | Lakehouse | Raw ingestion from ERP |
| `silver_orders_transform` | Notebook | Cleans and conforms order data |
| `gold_lakehouse` | Lakehouse | Star schema for reporting |
| `sales_model` | Semantic Model | DAX measures for Power BI |
| `pl_erp_bronze` | Pipeline | Daily load at 2 AM UTC |

**Data sources:**

- ERP system (SQL Server via gateway) - orders, customers, products
- SharePoint - monthly budget spreadsheet
- REST API - exchange rates (daily)

**Data flow:**

```
ERP -> pl_erp_bronze -> bronze_lakehouse -> silver_orders_transform -> gold_lakehouse -> sales_model -> Reports
```

**Business rules** *(critical for AI-assisted coding)*:

- Fiscal year starts April 1 (FY2025 = Apr 2024 - Mar 2025)
- Revenue = `quantity * unit_price - discount`
- Customer segments: Enterprise (>$1M annual), Mid-Market ($100K-$1M), SMB (<$100K)
- Currency: all amounts in USD after exchange rate conversion in silver layer
- Inactive customers: no orders in last 18 months - exclude from active metrics

**Sensitive data:**

- `customer_email`, `customer_phone` - PII, masked in gold layer
- `employee_id` - do not expose in reports
- Retention policy: raw data deleted after 7 years

---

### Project 2: *(Copy this template for additional projects)*

*(Delete this placeholder or duplicate the template above)*


In [ ]:
# --- Project -> Item Validation ---
# Maps project names to their expected workspace items.
# Run this cell to check which items exist and flag orphans.

PROJECT_ITEMS = {
    "Sales Analytics": [
        "bronze_lakehouse",
        "silver_orders_transform",
        "gold_lakehouse",
        "sales_model",
        "pl_erp_bronze",
    ],
    # Add more projects:
    # "Marketing Dashboard": ["marketing_lakehouse", "campaign_model"],
}

all_item_names = set(items["Display Name"].tolist())
assigned = set()

for project, expected_items in PROJECT_ITEMS.items():
    found = [i for i in expected_items if i in all_item_names]
    missing = [i for i in expected_items if i not in all_item_names]
    assigned.update(found)
    status = "\u2705" if not missing else "\u26a0\ufe0f"
    print(f"{status} {project}: {len(found)}/{len(expected_items)} items found")
    if missing:
        print(f"   Missing: {missing}")

orphans = sorted(all_item_names - assigned - {"workspace-context"})
if orphans:
    print(f"\n\u26a0\ufe0f Orphan items (not in any project): {orphans}")
else:
    print(f"\n\u2705 All items assigned to a project")


## Architecture / Data Flow

Describe how items in this workspace relate to each other.

```
Example:
Source Systems -> (Pipelines) -> Bronze Lakehouse -> (Notebooks) -> Silver Lakehouse -> (Notebooks) -> Gold Lakehouse -> Semantic Model -> Reports
```

| Item | Role | Notes |
|------|------|-------|
| `bronze_lakehouse` | Raw ingestion | Source system data, minimal transforms |
| `silver_lakehouse` | Cleaned & conformed | SCD Type 2, standardized column names |
| `gold_lakehouse` | Business aggregates | Star schema for reporting |
| `sales_model` | Semantic model | DAX measures, consumed by Power BI |


In [ ]:
# --- Schema Discovery (auto-discovered) ---
# Uses notebookutils.fs.ls() instead of fabric.list_tables() because
# list_tables() has a known bug with schema-enabled lakehouses.
#
# Logic: /Tables/ lists schemas (if schema-enabled) or tables directly.
#        /Tables/<schema>/ lists tables within that schema.

# Auto-detect lakehouses if not specified
LAKEHOUSE_NAMES = []  # e.g., ["bronze_lakehouse", "silver_lakehouse"]
if not LAKEHOUSE_NAMES:
    lh_items = items[items["Type"] == "Lakehouse"]
    LAKEHOUSE_NAMES = lh_items["Display Name"].tolist()
    print(f"Auto-detected lakehouses: {LAKEHOUSE_NAMES}")

lakehouse_schemas = {}  # {lh_name: {schema: [table_names]}}

for lh_name in LAKEHOUSE_NAMES:
    try:
        props = notebookutils.lakehouse.getWithProperties(lh_name)
        abfss = props["properties"]["abfsPath"]
        tables_path = f"{abfss}/Tables"
        print(f"\n=== {lh_name} ===")
        print(f"    ABFSS: {abfss}")

        top_level = notebookutils.fs.ls(tables_path)
        lh_schema_map = {}

        for entry in top_level:
            if entry.isDir:
                # Could be a schema folder or a table folder (non-schema lakehouse)
                sub_items = notebookutils.fs.ls(entry.path)
                sub_dirs = [s.name for s in sub_items if s.isDir]
                if sub_dirs:
                    # This is a schema folder containing tables
                    lh_schema_map[entry.name] = sub_dirs
                    print(f"    Schema '{entry.name}': {sub_dirs}")
                else:
                    # This is a table folder (no schemas)
                    lh_schema_map.setdefault("default", []).append(entry.name)

        if "default" in lh_schema_map:
            print(f"    Tables: {lh_schema_map['default']}")

        lakehouse_schemas[lh_name] = lh_schema_map

    except Exception as e:
        print(f"\n=== {lh_name} === (error: {e})")
        lakehouse_schemas[lh_name] = {}


## Schema Conventions

Document your team's schema standards here.

- **Table naming**: `snake_case` (e.g., `order_items`, `dim_customer`)
- **Date columns**: `*_date` for DATE type, `*_at` for DATETIME2
- **ID columns**: `*_id` suffix, always INT or BIGINT
- **Soft deletes**: `is_deleted` flag, never hard delete
- **Source columns**: Portuguese source columns aliased to English in silver layer


## Conventions & Standards

### Naming Conventions

| Item Type | Pattern | Example |
|-----------|---------|--------|
| Lakehouses | `{layer}_lakehouse` | `bronze_lakehouse` |
| Notebooks | `{layer}_{entity}_transform` | `silver_orders_transform` |
| Pipelines | `pl_{source}_{destination}` | `pl_erp_bronze` |
| Semantic Models | `{domain}_model` | `sales_model` |

### Branching Strategy

- `main` -> PROD (auto-deploy via deployment pipeline)
- `dev` -> DEV workspace (Git-connected)
- Feature branches -> personal workspaces

### Code Standards

- PySpark preferred over Spark SQL for transforms
- All notebooks must have a markdown cell at top explaining purpose
- Use `notebookutils.credentials` for secrets, never hardcode
- All code cells should be idempotent (safe to re-run)


## Gotchas & Lessons Learned

Hard-won lessons specific to this workspace. Add to this list when you discover something.

1. SQL endpoint for lakehouses is **read-only** - use Spark for writes
2. `orders` table has ~50M rows - always filter by `order_date` first
3. Deployment pipeline takes ~8 min for full sync - don't re-trigger
4. Use `-G` flag for Entra ID auth with sqlcmd
5. Token audience for Fabric REST API: `https://api.fabric.microsoft.com`
6. Token audience for OneLake: `https://storage.azure.com`


## Ownership & Contacts

| Item | Owner | Notes |
|------|-------|-------|
| `bronze_lakehouse` | @name | Source system integration |
| `silver_lakehouse` | @name | Transform logic |
| `gold_lakehouse` | @name | Star schema design |
| `sales_model` | @name | Semantic model + DAX measures |
| `pl_erp_bronze` | @name | Runs daily at 2 AM UTC |


## Changelog

Append major changes here. Most recent first.

- **YYYY-MM-DD**: Initial workspace-context notebook created


In [ ]:
# --- AI Assistant Context (structured JSON output) ---
# This cell outputs machine-readable context for AI coding assistants.
# An AI assistant can read this output to bootstrap a session.

import json

# === CURATED CONTEXT (update manually) ===
STAGE = "development"  # development | test | production
CONVENTIONS = {
    "naming": "snake_case",
    "date_columns": "*_date for DATE, *_at for DATETIME2",
    "id_columns": "*_id suffix, INT or BIGINT",
    "soft_deletes": True
}
GOTCHAS = [
    "Lakehouse SQL endpoint is read-only \u2014 use Spark for writes",
    "orders table: always filter by order_date (50M rows)",
    "Deployment pipeline takes ~8 min for full sync"
]

# === AUTO-DISCOVERED CONTEXT ===
context = {
    "workspace_id": workspace_id,
    "workspace_name": workspace_name,
    "last_refreshed": last_refreshed,
    "stage": STAGE,
    "items": {},
    "project_items": PROJECT_ITEMS,
    "lakehouses": {},
    "conventions": CONVENTIONS,
    "gotchas": GOTCHAS,
    "workspace_settings": {
        "capacity_id": ws_info.get("capacityId"),
        "capacity_region": ws_info.get("capacityRegion"),
        "onelake_endpoints": ws_info.get("oneLakeEndpoints", {}),
        "spark": {
            "default_pool": spark_info.get("pool", {}).get("defaultPool", {}).get("name"),
            "runtime_version": spark_info.get("environment", {}).get("runtimeVersion"),
            "session_timeout_min": spark_info.get("job", {}).get("sessionTimeoutInMinutes"),
        },
        "git": {
            "provider": git_info.get("gitProviderDetails", {}).get("gitProviderType"),
            "org": git_info.get("gitProviderDetails", {}).get("organizationName"),
            "repo": git_info.get("gitProviderDetails", {}).get("repositoryName"),
            "branch": git_info.get("gitProviderDetails", {}).get("branchName"),
        },
    },
}

# Populate items by type (includes Description from item settings)
for item_type in items["Type"].unique():
    type_items = items[items["Type"] == item_type]
    context["items"][item_type] = [
        {
            "name": row["Display Name"],
            "id": row["Id"],
            "description": row.get("Description", "") or ""
        }
        for _, row in type_items.iterrows()
    ]

# Populate lakehouse schemas and tables
for lh_name, schema_map in lakehouse_schemas.items():
    context["lakehouses"][lh_name] = schema_map

print(json.dumps(context, indent=2))


In [ ]:
# --- Optional: Save context to OneLake for programmatic access ---
# Uncomment the lines below to write the structured context as a JSON file.
# Useful for read-only workspaces (UAT/PROD) where a pipeline runs this notebook
# and external tools consume the JSON without running the notebook interactively.

# output_path = "/lakehouse/default/Files/WORKSPACE-CONTEXT.json"
# with open(output_path, "w") as f:
#     json.dump(context, f, indent=2)
# print(f"Context saved to: {output_path}")
